In [ ]:
import torch
from training_functions import train, evaluate_mae_mse
from mcnn import MCNN
import torch.nn as nn
import torch.optim as optim
from dataloader_mcnn import CrowdDataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset
import glob
import os
import numpy as np
import random
import time
import pandas as pd
from pytorch_msssim import ssim
import os
import random
import zipfile
from pathlib import Path

from torchmetrics.functional.image import (
    structural_similarity_index_measure as ssim
)
from rbm import RBM
import joblib
from sklearn.metrics import mean_squared_error
import seaborn as sns
from CSRnet import CSRNet
from torchvision import transforms
import torchvision.transforms.functional as F
SEED = 42

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CSRNet()
model.load_state_dict(torch.load(r'output\models\csrnet\shanghaia_converted.pth'))
model.eval()
model = model.to(device)

In [ ]:


def csrnet_preprocess(img):
    if isinstance(img, torch.Tensor):
        img = F.to_pil_image(img)

    if img.width > 1500 or img.height > 1500:
        img = img.resize((img.width // 2, img.height // 2))

    img = 255.0 * F.to_tensor(img)

    img[0] -= 92.8207477031
    img[1] -= 95.2757037428
    img[2] -= 104.877445883

    return img

### General tests

In [ ]:
main_folders = [
    r"data\ShanghaiTech_part_A\test", 
    r"data\ShanghaiTech_part_B\test",
    r"data\NWPU-Crowd\val",
    r"data\UCF-QNRF_ECCV18\test",
    r"data\jhu_crowd_v2.0\test",
    r"data\UCF_CC_50"
]
evaluations = []
for folder in main_folders:    
    test_img_root = os.path.join(folder, "images")
    test_map_root = os.path.join(folder, "maps")
    test_dataset = CrowdDataset(test_img_root, test_map_root, gt_downsample=2, resize=False, transform=csrnet_preprocess)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, pin_memory=True)
    mae, mse = evaluate_mae_mse(model, test_loader, device)
    rmse = mse**(1/2)
    evaluations.append({'dataset': folder.split('/')[-2] ,'mae': mae, 'rmse': rmse})

In [ ]:
df2 = pd.DataFrame(evaluations)
df2.to_csv(r"output\history\csrnet\evaluation.csv")

### Advanced tests

In [ ]:
main_folders = [
    r"data\ShanghaiTech_part_A\test", 
    r"data\ShanghaiTech_part_B\test",
    r"data\NWPU-Crowd\val",
    r"data\UCF-QNRF_ECCV18\test",
    r"data\jhu_crowd_v2.0\test",
    r"data\UCF_CC_50"
]
evaluations = []
model.eval()
with torch.no_grad():
    for folder in main_folders:    
        test_img_root = os.path.join(folder, "images")
        test_map_root = os.path.join(folder, "maps")
        test_dataset = CrowdDataset(test_img_root, test_map_root, gt_downsample=2, resize=False, transform=csrnet_preprocess)
        for img,gt_dmap in test_dataset:
            img = img.unsqueeze(0).to(device)
            pred = model(img).squeeze(0)
            del img
            pred_sum = pred.sum().item()
            del pred
            torch.cuda.empty_cache()
            gt = gt_dmap.sum().item()
            diff = pred_sum - gt
            diff_abs = abs(diff)
            perc = diff_abs / (gt+1e-9)
            evaluations.append({'dataset': folder.split('/')[-2] ,'pred': pred_sum, 'gt': gt, 'diff': diff, 'abs_err': diff_abs, 'perc': perc})

In [ ]:
df = pd.DataFrame(evaluations)
df.to_csv(r"output\history\csrnet\evaluation_extended.csv")

In [ ]:
df2 = pd.read_csv(r'output\history\csrnet\evaluation_extended.csv')
df2 = df2[df2['gt'] != 0]
df2.groupby('dataset').mean()

In [ ]:
plt.scatter(df2['pred'], df2['diff'])
plt.title('CSRNet')
plt.xlabel('Predicted')
plt.ylabel('Difference')

In [ ]:
plt.scatter(df2['gt'], df2['pred'])
plt.plot([0,max(df2['gt'])], [0, max(df2['gt'])], color='red')
plt.title('CSRNet')
plt.xlabel('Ground truth')
plt.ylabel('Predicted')

In [ ]:
plt.scatter(df2['gt'], df2['perc'])
plt.title('CSRNet')
plt.xlabel('Ground truth')
plt.ylabel('Error')

In [ ]:
def bins(x):
    if x <= 50:
        return 'small'
    if x <= 200:
        return 'medium'
    if x <= 1000:
        return 'high'
    else:
        return 'large'

In [ ]:
df2 = pd.read_csv(r'output\history\csrnet\evaluation_extended.csv')
df2 = df2[df2['gt'] != 0]
df2['bins'] = df2['gt'].apply(lambda x: bins(x))
df2['rmse'] = (df2['pred'] - df2['gt'])**2
df2.drop(columns='dataset', inplace=True)
df2 = df2.groupby(['bins']).mean()
df2['rmse'] = df2['rmse']**(1/2)
df2

In [ ]:
df2 = pd.read_csv(r'output\history\csrnet\evaluation_extended.csv')
df2 = df2[df2['gt'] == 0]

In [ ]:
bins = np.logspace(
    0,
    np.log10(np.abs(df2['pred']).max()),
    15
)

plt.hist(
    np.abs(df2['pred']),
    bins=bins,
    edgecolor='black',
    linewidth=1.2
)

plt.xscale('log')

plt.xlabel("Log Prediction")
plt.ylabel("Count")
plt.title("CSRNet (Images with zero ground truth)")

plt.show()

### Denisty maps

In [ ]:
main_folders = [
    r"data\ShanghaiTech_part_A\test", 
    r"data\ShanghaiTech_part_B\test",
    r"data\NWPU-Crowd\val",
    r"data\UCF-QNRF_ECCV18\test",
    r"data\jhu_crowd_v2.0\test",
    r"data\UCF_CC_50"
]
evaluations = []

model.eval()

with torch.no_grad():
    for folder in main_folders:

        test_img_root = os.path.join(folder, "images")
        test_map_root = os.path.join(folder, "maps")

        test_dataset = CrowdDataset(
            test_img_root,
            test_map_root,
            gt_downsample=4,
            resize=False,
            transform=csrnet_preprocess
        )

        for img, gt_dmap in test_dataset:

            img = img.unsqueeze(0).to(device)
            pred = model(img)

            if gt_dmap.dim() == 2:
                gt_dmap = gt_dmap.unsqueeze(0).unsqueeze(0)
            elif gt_dmap.dim() == 3:
                gt_dmap = gt_dmap.unsqueeze(0)

            pred = pred.detach()

            gt_dmap = (gt_dmap/(gt_dmap.sum()+1e-8)).to(device)
            pred = (pred/(pred.sum()+1e-8)).to(device)

            vmin = min(gt_dmap.min(), pred.min())
            vmax = max(gt_dmap.max(), pred.max())
            ssim_val = ssim(
                pred,
                gt_dmap,
                data_range=vmax-vmin
            )

            evaluations.append({
                "dataset": folder.split('/')[-2],
                "ssim": ssim_val.item()
            })

            del img, pred, gt_dmap
            torch.cuda.empty_cache()

In [ ]:
df = pd.DataFrame(evaluations)
df.to_csv(r'output\history\csrnet\evaluation_denistymaps.csv')

In [ ]:

main_folders = [
    r"data\ShanghaiTech_part_A\test", 
    r"data\ShanghaiTech_part_B\test",
    r"data\NWPU-Crowd\val",
    r"data\UCF-QNRF_ECCV18\test",
    r"data\jhu_crowd_v2.0\test",
    r"data\UCF_CC_50"
]

dataset_names = [
    "ShanghaiTech A",
    "ShanghaiTech B",
    "NWPU-Crowd",
    "UCF-QNRF",
    "JHU-Crowd",
    "UCF-CC-50"
]

output_dir = Path(r"output\history\csrnet")
output_dir.mkdir(parents=True, exist_ok=True)

def normalize_map(x):
    x = x.astype(np.float32)
    return x / (x.sum() + 1e-8)

model.eval()
set_seed(124)

with torch.no_grad():

    for folder, dataset_name in zip(main_folders, dataset_names):

        test_img_root = os.path.join(folder, "images")
        test_map_root = os.path.join(folder, "maps")

        test_dataset = CrowdDataset(
            test_img_root,
            test_map_root,
            gt_downsample=8,
            resize=False,
            transform=csrnet_preprocess
        )

        idxs = random.sample(range(len(test_dataset)), 2)

        fig, axes = plt.subplots(
            nrows=2,
            ncols=5,
            figsize=(20, 8),
            constrained_layout=True
        )

        fig.suptitle(
            dataset_name,
            fontsize=18,
            fontweight="bold"
        )

        col_titles = [
            "Image",
            "Ground Truth",
            "Predicted",
            "Normalized GT",
            "Normalized Pred"
        ]

        for c, title in enumerate(col_titles):
            axes[0, c].set_title(title, fontsize=12)

        for row, idx in enumerate(idxs):

            img, gt_dmap = test_dataset[idx]

            img_vis = img.permute(1, 2, 0).cpu().numpy()

            pred = model(
                img.unsqueeze(0).to(device)
            )

            if gt_dmap.dim() == 2:
                gt_t = gt_dmap.unsqueeze(0).unsqueeze(0)

            elif gt_dmap.dim() == 3:
                gt_t = gt_dmap.unsqueeze(0)

            else:
                gt_t = gt_dmap

            pred_t = pred.detach()

            gt_t = gt_t.to(device)
            pred_t = pred_t.to(device)

            raw_data_range = (
                gt_t.max() - gt_t.min() + 1e-8
            )

            raw_ssim = ssim(
                pred_t,
                gt_t,
                data_range=raw_data_range
            ).item()

            gt_np = gt_t.squeeze().cpu().numpy()
            pred_np = pred_t.squeeze().cpu().numpy()

            gt_norm = normalize_map(gt_np)
            pred_norm = normalize_map(pred_np)

            gt_norm_t = (
                torch.from_numpy(gt_norm)
                .float()
                .unsqueeze(0)
                .unsqueeze(0)
                .to(device)
            )

            pred_norm_t = (
                torch.from_numpy(pred_norm)
                .float()
                .unsqueeze(0)
                .unsqueeze(0)
                .to(device)
            )
            vmin = min(
                gt_norm.min(),
                pred_norm.min()
            )

            vmax = max(
                gt_norm.max(),
                pred_norm.max()
            )

            norm_data_range = (
                gt_norm_t.max()
                - gt_norm_t.min()
                + 1e-8
            )

            norm_ssim = ssim(
                pred_norm_t,
                gt_norm_t,
                data_range=norm_data_range
            ).item()

            print(
                f"{dataset_name} | "
                f"Image {idx} | "
                f"Raw SSIM: {raw_ssim:.4f} | "
                f"Normalized SSIM: {norm_ssim:.4f}"
            )

            axes[row, 0].imshow(img_vis)
            axes[row, 0].axis("off")
            
            axes[row, 1].imshow(gt_np, cmap="jet")
            axes[row, 1].axis("off")
            
            axes[row, 2].imshow(pred_np, cmap="jet")
            axes[row, 2].axis("off")
            
            axes[row, 2].text(
                 0.5, -0.08,
                f"SSIM = {raw_ssim:.4f}",
                transform=axes[row, 2].transAxes,
                ha="center",
                va="bottom",
                fontsize=13,
                fontweight="bold",
                color="white",
                bbox=dict(
                    facecolor="black",
                    alpha=0.75,
                    pad=4
                )
            )
            
            axes[row, 3].imshow(
                gt_norm,
                cmap="jet",
                vmin=vmin,
                vmax=vmax
            )
            axes[row, 3].axis("off")
            
            axes[row, 4].imshow(
                pred_norm,
                cmap="jet",
                vmin=vmin,
                vmax=vmax
            )
            axes[row, 4].axis("off")
            
            axes[row, 4].text(
                 0.5, -0.08,
                f"SSIM = {norm_ssim:.4f}",
                transform=axes[row, 4].transAxes,
                ha="center",
                va="bottom",
                fontsize=13,
                fontweight="bold",
                color="white",
                bbox=dict(
                    facecolor="black",
                    alpha=0.75,
                    pad=4
                )
            )

        plt.show()



zip_path = r"output\history\csrnet\crowd_ssim_results.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file in output_dir.glob("*.png"):
        zipf.write(
            file,
            arcname=file.name
        )

In [ ]:
names = {'ShanghaiTech_part_A': 'ShanghaiTech A',
          'ShanghaiTech_part_B': 'ShanghaiTech B',
          'NWPU-Crowd': 'NWPU-Crowd',
          'UCF-QNRF_ECCV18': 'UCF-QNRF',
          'jhu_crowd_v2.0': 'JHU-Crowd',
          'ucf-cc-50': 'UCF-CC-50'}

In [ ]:
df = pd.read_csv(r'output\history\csrnet\evaluation_densitymap.csv')
df.drop(columns='Unnamed: 0', inplace=True)
df['dataset'] = df['dataset'].apply(lambda x: names[x])
df.groupby('dataset').agg(['mean', 'std', 'min', 'max'])['ssim_norm']

In [ ]:
sns.boxplot(data=pd.DataFrame(df), y='ssim', hue='dataset')
plt.title('SSIM of Density Maps by Dataset')
plt.xlabel('Dataset')
plt.ylabel('SSIM')
plt.show()

In [ ]:
sns.boxplot(data=pd.DataFrame(df), y='ssim_norm', hue='dataset')
plt.title('SSIM of Density Maps by Dataset')
plt.xlabel('Dataset')
plt.ylabel('SSIM')
plt.show()

### Boltzman Machines

In [ ]:
data = np.load(r'output\models\BoltzmanCSRNet\predictions_tests.npy')
tensor_test_input = torch.from_numpy(data).to(dtype = torch.float64)

In [ ]:
machine1 = torch.load(r'output\models\BoltzmanMCNN\machine1.pt', weights_only=False)
machine2 = torch.load(r'output\models\BoltzmanMCNN\machine2.pt', weights_only=False)
machine3 = torch.load(r'output\models\BoltzmanMCNN\machine3.pt', weights_only=False)

In [ ]:
tensor1 = machine1.draw_hidden(tensor_test_input)
tensor2 = machine2.draw_hidden(tensor1)
tensor3 = machine3.draw_hidden(tensor2)

In [ ]:
X_test = tensor3.detach().cpu().numpy()
y_test = pd.read_csv(r'output\models\BoltzmanCSRNet\metadata_test.csv')['gt_sum'].to_numpy()
dataset_names = pd.read_csv(r'output\models\BoltzmanCSRNet\metadata_test.csv')['dataset'].to_numpy()

In [ ]:
models_dir = Path(r"output\models\BoltzmanCSRNet")
output_dir = Path(r"output\history\boltzmann_csrnet")
output_dir.mkdir(exist_ok=True)

model_files = {
    "decision_tree": models_dir / "decision_tree_csrnet.joblib",
    "linear_regression": models_dir / "linear_regression_csrnet.joblib",
    "random_forest": models_dir / "random_forest_csrnet.joblib",
    "svm": models_dir / "svm_csrnet.joblib",
}

X_test_np = np.asarray(X_test)
y_test_np = np.asarray(y_test)

dataset_names = np.asarray(dataset_names)
y_test_np = y_test_np.reshape(-1)

for model_name, model_path in model_files.items():

    print(f"{model_name}")
    model = joblib.load(model_path)
    pred = model.predict(X_test_np)
    pred = np.asarray(pred).reshape(-1)
    diff = pred - y_test_np
    abs_err = np.abs(diff)
    rmse = np.sqrt(np.mean(diff ** 2))
    perc = np.full_like(abs_err, np.nan, dtype=float)
    nonzero = y_test_np != 0
    perc[nonzero] = abs_err[nonzero] / np.abs(y_test_np[nonzero])
    results = pd.DataFrame({
        "dataset": dataset_names,
        "pred": pred,
        "gt": y_test_np,
        "diff": diff,
        "rmse": rmse,
        "abs_err": abs_err,
        "perc": perc
    })

    output_path = output_dir / f"{model_name}_test_results.csv"
    results.to_csv(output_path, index=False)